### `DDFM: Denoising Diffusion Model for Multi-Modality Image Fusion`

 :::
 - La fusion d'images multi-modalités combine différentes modalités pour obtenir des images fusionnées. 
 - Les images conservent les caractéristiques complémentaires (détails de texture, points fonctionnels importants). 
 - Les méthodes génératives basées sur les GAN sont sources de difficultés. 
 - Celles-ci sont résolues par le nouvel algorithme de fusion qui utilise le modèle de diffusion probabiliste. 
 - Le processus formulé par cette méthode se réfère a un problème de génération conditionnelle. 
 - Le problème est divisé en sous-problèmes, répondu à l'aide de l'approche bayésienne hiérarchique. 
 - Intégrer la solution dans les itérations de diffusion, permet la génèration d'images de haute qualité.
 - L'algorithme utilise des priorités de génération d'images naturelles et des informations inter-modales.
:::

:::
__`CONTEXTE IMAGE`__

`IRM (Imagerie par Résonance Magnetique)` 
L’IRM est particulièrement utile pour les tissus mous : cerveau, moelle, muscles, ligaments, organes du pelvis (utérus, vessie, etc.), où elle donne un contraste bien meilleur que le scanner ou la radio.
- C'est une technique d’imagerie médicale qui utilise un champ magnétique puissant et des ondes radio.
- Obtenue par imagerie par résonance magnétique, 
- Elle produit des vues très détaillées de l’intérieur du corps, en 2D ou 3D.

`US GT (Ultrasound Ground Truth)`
Ce sont des images d’échographie issus d'ondes sonores HF envoyées dans le corps avec une sonde.
Dans notre contexte ce sont des images de référence synthétiques (théorique ou idéale).
- US GT, image Ultra Sound « idéale » (ground truth) simulée à partir de l’IRM avec un modèle polynomial. 
- US observation, version bruitée d'US GT, avec un bruit speckle modélisé par du log‑Rayleigh. 
- US observation a pour objectif d'imiter le bruit caractéristique des vraies images US.
:::
:::
__`CONTEXTE PROJET`__.  
Dans notre cas les images IRM sont les images de référence riches en contraste anatomique (formes, structures internes), que l’on fusionne avec les images US pour bénéficier à la fois des détails IRM et de la localisation des bords donnée par l’échographie.

1. Les images US sont mathématiquement dérivées de l’IRM puis fusionnées avec l’IRM à l'aide d'un modèle polynomial, puis bruitées.

On ne dispose pas du couple IRM + US d'un même patient parfaitement aligné, on simule le cas clinique :
 - On part d’une IRM (image riche en contraste de tissus).
 - On génère un US GT depuis l’IRM via un modèle polynomial qui combine intensité et gradient de l’IRM.
 - On ajoute du bruit (gaussien sur l’IRM, log‑Rayleigh sur l’US) pour obtenir les observations IRM et US.


2. La fusion cherche à illustrer
Bien que les US viennent de l’IRM, le but est d’illustrer le principe de la fusion IRM+US :
 - L’IRM apporte un contraste global et une bonne visualisation des structures internes.
 - L’US (même simulée) apporte des bords nets / pics d’intensité, mais avec un bruit speckle important.
 - L’algorithme de fusion (PALM + débruitage DnCNN) essaie de reconstruire une image qui combine :
  - le contraste IRM,
  - la localisation précise des interfaces/bords venant de l’US.

Un vrai cas clinique, nécessiterait de disposer d'une vraie IRM et d'une vraie US non généré.
Le comportement du modèle est plus facile à analyser, à valider à partir d’un scénario synthétique contrôlé.
:::

:::
#### `Présentation du projet sujet 15 :`
`L'objectif du projet est de:`
 
 - _Lire, comprendre, analyser l’article et les codes associés._
 - _Reproduire des résultats du papier._
 - _Adapter légèrement les codes_ 
 - _Tester la méthode sur nos images ultrasonores et IRM (disponibles dans le répertoire /images)_ 

`Note :` 
 - Consulter le README.doc dans /images pour obtenir des informations sur le jeux de données à tester.
 - Comparer les résultats obtenus avec ceux obtenus en exécutant le fichier “Demo.m”.
 - Utiliser les données Data1, Data2 et Data3 fournis pour générer les ensembles de données 2 et 3.         
 - L'exécution des fonctions `Synthetic1.m` et `Synthetic2.m` produit les ensembles de données 2 et 3.
::: 
:::
__`Synthétic1 chaîne de simulation + fusion sur image IRM synthétique issue d'un file.mat`.__
  1. Part d’une IRM synthétique (init_mi256_2.mat),
  2. Génère une image US « parfaite » via un modèle polynomial,
  3. Crée des observations bruitées IRM + US,
  4. Fusionne ces observations via PALM en utilisant un pré‑débruitage DnCNN et une estimation du lien IRM↔US
:::
:::
__`Synthetic2.m est un pipeline complet de simulation + fusion sur une IRM réelle`__
  1. IRM réelle → US synthétique « parfaite »
  2. US synthétique « parfaite » → versions bruitées IRM/US 
  3. Versions bruitées IRM/US → fusion par PALM aidée d’un débruitage deep learning (DnCNN).
:::

#### __`Synthétic1.m`__
:::
__`1. Chargement de l’IRM « ground truth » et génération de l’US idéale`__
- Charge images/init_mi256_2.mat et récupère irm_gt = init_mi256_2; (IRM de référence).
- Définit un vecteur de coefficients c de taille 10, et génère une image US « ground truth » via us_gt = poly(irm_gt, c);
- La fonction poly(x1,c) calcule le gradient de x1 (Jx, Jy, gradY), 
- puis construit x2 comme un polynôme de x1 et de gradY (termes en $x,x^2,x^3,gradY,gradY⋅x,gradY^2…)$ → 
- ceci modélise une relation IRM → US tenant compte à la fois des intensités et des contours.
:::
:::
__`2. Simulation des images observées (bruitées)`__
__IRM observée irm :__
 - floutage gaussien : irm1_blurred = imgaussfilt(irm_gt,4);
 - décimation neutre (ici juste affectation) : irm1_resized = irm1_blurred;
 - ajout de bruit gaussien additif de variance contrôlée :

__Avant traitement (ground truth) :__
 - irm_gt : IRM de référence, affichée et sauvegardée en images/irm_gt.png.
 - us_gt : US synthétique de référence, affichée et sauvegardée en images/us_gt.png.

__Après « dégradation » / observation :__
 - irm : IRM floutée + bruit gaussien, affichée et sauvegardée en images/irm.png.
 - us : US avec bruit de type log‑Rayleigh, affichée et sauvegardée en images/us.png.
   - sigma_noise = 0.02;
   - irm = irm1_resized + sigma_noise*randn(size(irm1_resized));

__US observée us :__
 - dimension n = size(irm_gt);
 - ajout de bruit de type log‑Rayleigh : (Rayleigh/log‑Rayleigh typique du speckle dans les images US.)
  - gama = 0.21;
  - us = us_gt + gama*log(raylrnd(1.1, n(1), n(2)));

__Quatre figures sont affichées__
 - Ground truth MRI → irm_gt → images/irm_gt.png
 - Ground truth US → us_gt → images/us_gt.png
 - MRI observation → irm → images/irm.png
 - US observation → us → images/us.png
 - Donc on voit bien avant / après dégradation pour les deux modalités. 
:::
:::
__`3. Fusion par PALM avec pré‑débruitage DnCNN`__
Appelle estimate_c; qui estime les coefficients du modèle de lien IRM↔US à partir des observations (renvoie cest, pris en valeur absolue c = abs(cest);).

__Normalise les images observées :__
 - ym = double(irm) / max(irm(:));    % IRM normalisée
 - yu = double(us)  / max(us(:));     % US normalisée

__Applique un débruitage profond sur l’US :__
 - net = denoisingNetwork('DnCNN');
 - xu0 = denoiseImage(yu, net); → xu0 est une version débruitée de l’US, utilisée comme point de départ.

__Passe ensuite dans l’algorithme de fusion PALM :__
 - [x2] = FusionPALM(ym, xu0, c, tau1, tau2, tau3, tau4, true);
 - avec des pas tau1..tau4 adaptés → 
 - x2 est l’image fusionnée qui vise à combiner :
   - le contraste et la structure globale de l’IRM,
   - les bords/bords fins fournis par l’US.
:::
:::
__`4. Analyse des intensités sur une colonne`__
Choisit une colonne num_col = 265 et extrait, sur les lignes 100:300, les profils d’intensité :
- col_us      = yu(100:300, num_col);
- col_irm     = ym(100:300, num_col);
- col_fusion  = x2(100:300, num_col);

__Trace les trois profils normalisés sur le même graphe :__
 1. IRM (k),
 2. US (g),
 3. Fusion (r) → Illustration de l’idée : 
 - IRM = bon contraste entre tissus mais frontière floue, 
 - US = bon pic au bord mais bruitée, fusion = combine les deux.
 - Visualise aussi la colonne choisie sur l’image US (yu) en ajoutant un offset sur la colonne pour la mettre en évidence.
:::
:::
__`5. Fonction poly (lien IRM → US)`__:
 - _poly(x1,c)_ calcule le gradient de l’IRM,
 - Elle produit une image US synthétique comme polynôme en x1 et en gradY.
 - C’est le modèle de génération d’US à partir d’IRM 
 - Il sert à la fois pour fabriquer us_gt et (via estimate_c) pour la fusion.
:::  

#### __`Synthetic2.m`__ 
 - c'est un pipeline complet de simulation + fusion sur une IRM réelle :`
 - IRM réelle → US synthétique « parfaite » → versions bruitées IRM/US → fusion par PALM aidée d’un débruitage deep learning (DnCNN).

 - Charge une IRM réelle, lit images/irm_simu.PNG, 
 - conserve le canal 1 (grayscale) et normalise entre 0 et 1.
 - Affiche cette image comme MRI ground truth.
 - Génère une image US « idéale » à partir de l’IRM

Utilise la fonction locale Link_poly(x1,c) (même structure que poly) pour appliquer un modèle polynomial de lien IRM→US basé sur les intensités et le gradient. Le résultat us1_gt est l’US « ground truth » correspondante à l’IRM, affichée comme US GT.

Crée des observations bruitées (IRM et US)
IRM observée irm1 :
 - flou gaussien imgaussfilt(irm1_gt,4) (dégradation spatiale),
 - ajout de bruit gaussien additif de variance sigma_noise = 0.026.

US observée us1 :
 - ajout de bruit de type log‑Rayleigh gamma1*log(raylrnd(1,n11,n21)), pour simuler le speckle/compression typique des images US.

Les deux sont affichées comme MRI observation et US observation.

Fusionne les images observées IRM/US
 - Fixe irm = irm1, us = us1.
 - Estime les coefficients de lien c via estimate_c (apprentissage du modèle IRM↔US sur les observations).
 - Normalise les images ym (IRM) et yu (US).
 - Applique un débruitage profond DnCNN à l’US observée :

matlab pour obtenir une US débruitée de départ.
 - net = denoisingNetwork('DnCNN');
 - xu0 = denoiseImage(yu, net);

Passe le couple (IRM observée, US débruitée) + coefficients c dans FusionPALM avec des pas spécifiques tau1..tau4 pour produire l’image fusionnée x2, qui combine contraste IRM et bordures US (fusion basée sur PALM).

Fonction interne Link_poly, calcule le gradient de l’IRM (gradY) et construit l’US synthétique comme un polynôme en x1 et gradY (termes en $x,x^2,x^3,gradY,gradY⋅x,…)$, ce qui encode la relation structurelle entre IRM et US.

#### `Résumé de l'article:` 
:::
La fusion d’images multi‑modalités consiste à prendre plusieurs types d’images (modalités par exemple scanner CT, IRM, PET, infrarouge, visible) d’une même scène pour un même patient et de les combiner afin de produire une seule image fusionnée qui soit en mesure de conserver :

 - les informations “fonctionnelles” dites de contraste (zones d’activité métabolique en PET, régions chaudes en thermique, zones fortement vascularisées, etc.) mises en avant par certaines modalités.

 - les détails fins de structure et de texture (contours nets des organes, structures osseuses en CT, textures des tissus en IRM, détails visuels en image visible), fournis par d’autres modalités.

 Parmi les sous-domaines de la fusion d’images, la fusion infrarouge–visible (IVF) et a fusion d’images médicales (MIF) sont particulièrement difficiles en contexte multi-modal, car il faut à la fois modéliser les interactions entre modalités et préserver les informations essentielles de tous les capteurs impliqués.  
 
 Plus précisément, en IVF fusion infrarouge–visible, l’image fusionnée doit conserver à la fois le rayonnement thermique issu de l’infrarouge et les détails de texture issus du visible, afin de pallier la sensibilité du visible aux conditions d’illumination et le caractère bruité et basse résolution de l’infrarouge.  
 
 De son côté, la fusion d’images médicales  MIF permet d’aider au diagnostic et au traitement en combinant plusieurs modalités d’imagerie médicale pour localiser précisément des anomalies. 

`L'auteur s'affranchit des difficultés relatives aux méthodes génératives basées sur les GAN duent aux: ` 
 - __Entraînement instable__ : équilibre délicat générateur/discriminateur, mode collapse, dépendance forte aux hyper‑paramètres.
 - __Manque d’interprétabilité__ : la génération est un “black box mapping” bruit → image, difficile à relier à une procédure probabiliste claire ou à un schéma de reconstruction.

`Il utilise l'algorithme de fusion fondé sur le modèle de diffusion de débruitage probabiliste (DDPM) de type DDFM pour fusion multi‑modalité)et de conserver les puissants priori génératifs`.
Il formalise la fusion d’images comme un problème de génération conditionnelle : 
 - ce qui revient à générer l’image fusionnée en conditionnant sur les images sources (IR + visible, etc.).
 - réutilise un DDPM pré‑entraîné de façon inconditionnelle comme priori naturel aux connaissances préalables sur la forme que doivent avoir des images naturelles, apprises par un modèle génératif (ici un DDPM) et réutilisées comme contrainte forte dans la fusion.
- puis injectent l’information des sources dans le processus de sampling (via une formulation conditionnelle / bayésienne, EM, etc.).

__`L’idée :`__
 - Est de profiter du priori naturel appris par le DDPM pour garantir que l’image fusionnée reste sur la “variété des images réalistes”, tout en guidant la génération pour respecter les contenus de chaque modalité.

 - La tâche de fusion est formulée comme un problème de génération conditionnelle dans le cadre d’échantillonnage DDPM, qui est en outre décomposé en un sous‑problème de génération inconditionnelle et un sous‑problème de maximum de vraisemblance.  

__`1. “La fusion est un problème de génération conditionnelle`__
- échantillonner dans la loi a posteriori $p(F∣X_1,X_2,...)$ au moyen d’un DDPM,
- soit, faire du sampling conditionnel avec un modèle de diffusion (la génération dépend explicitement des images à fusionner).   
__`2. Décomposition en deux sous‑problèmes`__
`a) Sous‑problème de génération inconditionnelle`
 - On veut générer l’image fusionnée F en conditionnant sur les images sources  $X_1,X_2$, (IR, visible, CT, MRI, etc.).
 - A l'aide  d'un DDPM, entraîné de manière inconditionnelle sur des images naturelles/fusionnées (sans lui donner les modalités en entrée).
 - Le DDPM sait générer des images plausibles selon un a priori d’images naturelles p(F) : textures réalistes, contours cohérents, etc.
 - Ce bloc ne “voit” pas directement les images sources pendant l’entraînement de base : il fournit juste le générateur avec prior.

`b) Sous‑problème de maximum de vraisemblance`
 - Reste à imposer que l’image générée soit cohérente avec les images sources, pour cela, il introduit un modèle de vraisemblance $p(F∣X_1,X_2,...)$.  
 - Le modèle dit que si l’image fusionnée est F, à quel point est‑il probable d’observer les images sources $X_i$
 - Ils formulent alors un problème de maximum de vraisemblance (ou de likelihood rectification) :
 - Parmi les images plausibles selon le DDPM, privilégier celles qui expliquent le mieux les données sources (maximiser $p(F∣X_1,X_2,...))$
 - Comme cette vraisemblance n’est pas explicite, ils la formulent dans un cadre bayésien hiérarchique avec variables latentes 
 - La résolvent via un algorithme EM, puis injectent le résultat (un terme de correction) dans la boucle de diffusion.

__`3. Vue d’ensemble intuitive`__
 - Le DDPM inconditionnel sais à quoi ressemblent des images fusionnées réalistes en général” → il fournit un prior p(F).
 - Le sous‑problème de maximum de vraisemblance parmi les images réalistes, choisis celles qui collent le mieux aux modalités d’entrée” → 
 - Il impose la cohérence avec les sources via $p(X∣F)$
 - La génération conditionnelle finale p(F∣X) est obtenue en combinant les deux prior du DDPM + rectification par la vraisemblance, intégrée dans les étapes de sampling du diffusion model.


 - En intégrant la solution d’inférence dans l’itération d’échantillonnage de diffusion, notre méthode peut générer des images fusionnées de haute qualité, bénéficiant à la fois des a priori génératifs d’images naturelles et des informations inter‑modalités provenant des images sources.  

 - Notons que tout ce dont nous avons besoin est un modèle génératif pré‑entraîné inconditionnel, sans nécessiter de réglage fin.  

 - Nos nombreuses expériences montrent que notre approche produit des résultats de fusion prometteurs pour la fusion d’images infrarouge‑visible et la fusion d’images médicales.  
:::

#### `1. Formulation bayésienne de la fusion`

On note :  
- $F$ : l’image **fusionnée** que l’on cherche.  
- $X = (X_1, X_2, \dots, X_M))$ : les **images sources multi‑modales** (CT, MRI, PET, IR, visible, …).  

L’objectif est de caractériser la **distribution a posteriori** :  $p(F \mid X_1, \dots, X_M) = p(F \mid X)$
Par le théorème de Bayes :  $p(F \mid X) \propto p(F)\, p(X \mid F)$

où :  
- $p(F)$ est le **prior génératif** sur les images fusionnées (appris par le DDPM inconditionnel).  
- $p(X \mid F)$ est la **vraisemblance** : à quel point une image fusionnée $F$ est compatible avec les images sources $X$. 

#### `2. Rôle des deux “sous‑problèmes”`
##### `2.1 Sous‑problème de génération inconditionnelle`

On entraîne un DDPM pour approximer $p(F)$ :  
- Le DDPM ne donne pas une densité explicite, mais une procédure de **sampling** de cette loi a priori.  
- On apprend donc un modèle qui sait générer des $F$ plausibles, sans condition sur $X$.

##### `2.2 Sous‑problème de maximum de vraisemblance`

On introduit un modèle pour $p(X \mid F)$ :  
- Parmi les $F$ plausibles selon le prior $p(F)$, on veut ceux qui **maximisent** $p(X \mid F).  
- On peut voir cela comme un problème de **maximum de vraisemblance régularisé par le prior** :  

$hat{F} = \arg\max_F \; \big[ \log p(X \mid F) + \log p(F) \big]$
ce qui correspond au **maximum a posteriori (MAP)** sur $F$. 

Dans DDFM, cette combinaison $log p(F) + \log p(X \mid F)$ est réalisée **dans la boucle de diffusion** :  
- $log p(F)$ est implémenté par le DDPM (a priori génératif).  
- $\log p(X \mid F)$ est ajouté comme **terme de correction / guidance** (via un schéma de type EM / likelihood rectification).

#### `3. Intuition “sampling conditionnel”`

Pendant le sampling :  
- Si tu échantillonnes seulement selon $p(F)$, tu obtiens des images réalistes mais **pas forcément** cohérentes avec les modalités \(X\).  
- En ajoutant la vraisemblance, tu fais un sampling **guidé** par $p(X \mid F)$ :  
  - tu restes dans la variété des images plausibles (prior du DDPM),  
  - mais tu biases la trajectoire de diffusion vers des $F$ qui expliquent bien les données d’entrée.  

En résumé :  
- Le problème de fusion est de caractériser $p(F \mid X)$.  
- On le factorise comme $p(F \mid X) \propto p(F)\,p(X \mid F)$.  
- DDFM traite :  
  - $p(F)$ via un DDPM inconditionnel,  
  - $p(X \mid F)$ via un terme de vraisemblance intégré dans le sampling (EM / rectification), ce qui revient à approximer le MAP ou à s’approcher de la postérieure complète.


## `Introduction`
:::
1. `Les auteurs proposent comme modèle de fusion d’images l'utilisation des modèles de diffusion probabilistes de débruitage DDFM`. 

 - Cette méthode offre une représentation plus claire des objets, des scènes et trouve des applications variées, en détection de saillance, détection d’objets et segmentation sémantique, qu'avec modèles basés sur les GAN. 

 - Par rapport aux GAN, les DDPM n’ont pas besoin de discriminateur, ce qui atténue l'instabilité de l’entraînement, de plus le processus de génération DDPM est plus interprétable puisqu’il s'explicite comme un processus de débruitage. 

 - Les modèles DDPM génèrent des images de haute qualité en modélisant un processus de diffusion qui ramène progressivement une image bruitée vers une image propre, ils s’appuient sur un processus de diffusion de type Langevin, réalisant une série d’étapes de diffusion inverse pour produire des échantillons synthétiques convaincants. 

2. `La fusion d’images` 

- La fusion d'images consiste à combiner les informations importantes provenant de plusieurs images sources afin de produire une image fusionnée de haute qualité, et cela couvre différents types d’images d’entrée : images numériques classiques, images multi-modales et images de télédétection.  

 - Parmi les sous-domaines de la fusion d’images, la fusion infrarouge–visible (IVF) et la fusion d’images médicales (MIF) sont particulièrement difficiles en contexte multi-modal, car il faut à la fois modéliser les interactions entre modalités et préserver les informations essentielles de tous les capteurs impliqués.

 - Plus précisément, en IVF, l’image fusionnée doit conserver à la fois le rayonnement thermique issu de l’infrarouge et les détails de texture issus du visible, afin de pallier la sensibilité du visible aux conditions d’illumination et le caractère bruité et basse résolution de l’infrarouge.  

 - De son côté, la MIF permet d’aider au diagnostic et au traitement en combinant plusieurs modalités d’imagerie médicale pour localiser précisément des anomalies. 
 

3. `Les auteurs proposent donc DDFM, un modèle de fusion d’images basé sur DDPM`.  

Ils formulent la génération conditionnelle comme un problème d’échantillonnage sur la loi postérieure au sein du cadre DDPM, qui est ensuite décomposé en deux sous-problèmes :

  - un problème de diffusion pour la génération inconditionnelle qui impose un a priori d’images naturelles 
  - un problème de maximum de vraisemblance, utilisé pour rapprocher les images générées des images sources via une rectification par le terme de vraisemblance.  

Le premier impose un a priori d’images naturelles, tandis que le second est utilisé pour rapprocher les images générées des images sources via une rectification par le terme de vraisemblance.  

L’utilisation d’un DDPM pour modéliser l’a priori naturel permet de mieux générer les détails difficiles à contrôler par de simples fonctions de coût manuelles, ce qui produit des images plus convaincantes visuellement.  

En tant que méthode générative, DDFM permet une génération stable et contrôlable des images fusionnées, sans discriminateur, en appliquant cette rectification de vraisemblance à la sortie du DDPM.

6. `Les contributions sont résumées en trois points :` 
 - d’abord, un modèle d’échantillonnage postérieur basé sur DDPM pour la fusion multi-modale, composé d’un module de génération inconditionnelle et d’un module de rectification de vraisemblance conditionnelle, où l’échantillonnage des images fusionnées ne nécessite qu’un DDPM pré‑entraîné, sans fine‑tuning.  
 
 - Ensuite, comme la vraisemblance ne peut pas être écrite explicitement, la perte d’optimisation est reformulée comme un problème d’inférence probabiliste avec variables latentes, résolu par l’algorithme EM, puis intégré dans la boucle DDPM pour réaliser la génération conditionnelle.  
 
 - Enfin, de nombreuses expériences sur les tâches IVF et MIF montrent que DDFM produit de manière constante de bons résultats de fusion, en préservant efficacement structure et détails des images sources tout en respectant les exigences de fidélité visuelle.

####  __`2 En « vision score‑based SDE »`__ 
:::
En« vision score‑based SDE désigne la façon de voir les modèles de diffusion / score comme des équations différentielles stochastiques continues dans le temps.
 - `Idée de base :` On définit une SDE directe qui transforme progressivement les données réelles en bruit gaussien en injectant du bruit en continu. En sens inverse, on considère la SDE « backward in time » qui part du bruit et revient vers la distribution de données, en retirant progressivement le bruit.
 - `Rôle du « score :` La SDE inverse dépend uniquement du score, c’est‑à‑dire le gradient $\nabla_x \log p_t(x)$ de la log‑densité des données bruitées au temps $t$. On approxime ce champ de gradients par un réseau de neurones entraîné par score matching sur plusieurs niveaux de bruit.
 - `Conséquences pratiques :` En intégrant numériquement la SDE inverse avec ce score appris, on échantillonne des données (images, signaux, etc.) à partir d’un bruit gaussien. Ce cadre unifie les modèles de diffusion classiques et les score‑matching à plusieurs niveaux de bruit, et donne aussi une ODE déterministe équivalente (probability flow ODE) pour le calcul exact de vraisemblance.

Note : _probability flow ODE est une équation différentielle ordinaire déterministe qui produit les mêmes distributions marginales que la reverse SDE d’un modèle de diffusion, mais sans bruit aléatoire._ 

`« score‑based SDE » des modèles de diffusion se déroule, en trois étapes` 
- 1 définition du forward SDE,
- 2 définitiondu reverse SDE, 
- 3 l’objectif d’apprentissage du score.
:::  

:::
##### __`2.1) Forward SDE (équation 1)`__
- On part d’une image $x_0$ on la fait évoluer progressivement vers un bruit gaussien $x_T$ en ajoutant du bruit infinitésimal. 
- Cette évolution continue est modélisée par une SDE d’Itô :
  $$dx_t = -\frac{\beta(t)}{2} x_t\, dt + \sqrt{\beta(t)} \, dw_t$$
  - $dw_t$ est un processus de Wiener (Brownien) standard $(W_t)_{t \ge 0}$ s’il vérifie les propriétés :

     - $W_0 = 0$ presque sûrement.   
     - Il a des accroissements indépendants et stationnaires.    
     - Pour tout $0 \le s < t$, l’accroissement $W_t - W_s$ suit une loi normale $\mathcal N(0, t - s)$.    
     - Les trajectoires $t \mapsto W_t$ sont continues presque sûrement.  
  - $\beta(t)$ est un planning de bruit (noise schedule) choisi de façon à conserver la variance. 
- __Intuition__ : on a un drift qui ramène le signal vers 0 et un terme de diffusion qui ajoute du bruit gaussien dont l’intensité est contrôlée par $\beta(t)$.
:::
:::
##### __`2.2) Reverse SDE (équation 2)`__
- On montre que ce processus peut être inversé dans le temps. 
- Il existe aussi une SDE qui part d’un bruit presque gaussien $x_T$ et remonte vers une image « propre » $x_0$.  
- La SDE inverse s’écrit :
  
  $dx_t = \Big[ -\frac{\beta(t)}{2} x_t - \beta(t)\,\nabla_{x_t} \log p_t(x_t) \Big] dt + \sqrt{\beta(t)}\, dw_t$
  - $dw_t$ est maintenant le Brownien « à l’envers »
  - $\nabla_{x_t} \log p_t(x_t)$ `est le` **`score`** `de la distribution de` $x_t$. 

`1. Les termes un par un`
 - x : l’état (par ex. une image vectorisée) au temps t.
 - w : un mouvement brownien standard, donc $d_w$ est un bruit gaussien infinitésimal.
 - β(t) : «noise schedule» fonction > 0 règle l’intensité du bruit, la force de rappel au cours du temps.

`2. Le terme de drift` $$ -\frac{\beta(t)}{2} x \;-\; \beta(t)\,\nabla_x \log p_t(x).$$

- $-\frac{\beta(t)}{2} x$ : tire $x$ vers 0, ramène le signal vers l’origine.  
- $- \beta(t)\,\nabla_x \log p_t(x)$ : 
  - utilise le score $\nabla_x \log p_t(x)$ (gradient de la log‑densité des données bruitées au temps $t$) pour pousser les échantillons vers les régions de forte probabilité de la distribution cible.

_Intuition_ :  
- le premier terme « recentre » globalement,  
- le second « sculpte » la trajectoire pour suivre la forme de la vraie distribution des données.

`3. Le terme de diffusion`

Le bruit est :$\sqrt{\beta(t)}\,dw$ 
- Il ajoute un bruit gaussien d’amplitude contrôlée par $\sqrt{\beta(t)}$.
- Même en reverse‑time, on garde un terme stochastique
- on génère en faisant suivre à $x_t$ des trajectoires aléatoires guidées par le drift précédent.
:::

:::
##### __`2.3) Apprentissage du score (équation 3)`__

Le gradient du log de la densité, s'appelle le score d’une distribution, c’est un vecteur de même dimension que x qui indique, au point x, dans quelle direction la densité augmente le plus vite et à quelle vitesse.

 - Si $p(x)$ est une densité de probabilité, son score est $\nabla_{x_t} \log p_t(x_t)$

Le score oracle conditionnel est « le vrai score mathématique » de la distribution conditionnelle $p_{t}^0(x_t∣x_0)$, celui que l’on devrait connaître si on avait accès à la loi exacte.

 - On considère la loi de $x_t$ conditionnellement à l’image propre $x_0$ au temps t : $p_{t}^0(x_t∣x_0)$
 - Le score oracle conditionnel est donc $\nabla_{x_t} \log p_{t}^0(x_t∣x_0)$
 - $\nabla_{x_t} \log p_t(x_t)$ est inconnu, donc on l’approxime par un réseau $s_\theta(x_t, t)$, appelé *score network*.

__`On entraîne ce réseau par « denoising score matching » `:__ 
 - on sait que, conditionnellement à $x_0$, la distribution de $x_t$ est $p_t^0(x_t \mid x_0)$, 
 - on calcule le vrai score conditionnel $\nabla_{x_t} \log p_t^0(x_t \mid x_0)$ de la distribution issue du forward process.

L’objectif consiste à minimiser l’erreur quadratique moyenne entre le score prédit et ce score « oracle » conditionnel, en moyennant sur le temps et sur les données. Cette expression est la `loss d’apprentissage du score network` dans les score-based diffusion models: $\nabla_{x_t} \log p_t^0(x_t \mid x_0)$
$$
\mathbb{E}_t\,\mathbb{E}_{x_0}\,\mathbb{E}_{x_t \mid x_0}
\left\| s_\theta(x_t, t) - \nabla_{x_t} \log p_t^0(x_t \mid x_0) \right\|_2^2
$$

`Ce que veulent faire les auteur`
Ils veulent entraîner un réseau $s_\theta(x_t, t)$ à approximer le `score vrai`
$\nabla_{x_t} \log p_t^0(x_t \mid x_0)$, c’est‑à‑dire le gradient du log de la densité de $x_t$ sachant l’image propre $x_0$. 

`Lecture terme par terme`
- $\mathbb{E}_t$ : on moyenne sur le temps $t$, tiré uniformément sur $[0,T]$. 
- $\mathbb{E}_{x_0}$ : on moyenne sur les données réelles $x_0$ (échantillons du dataset). 
- $\mathbb{E}_{x_t \mid x_0}$ : pour un $x_0$ et un $t$ donnés, on moyenne sur les réalisations bruitées $x_t$ obtenues par le forward process (ajout de bruit gaussien suivant la SDE).

À l’intérieur, on a :
$$
\left\| s_\theta(x_t, t) - \nabla_{x_t} \log p_t^0(x_t \mid x_0) \right\|_2^2
$$
qui est simplement la **MSE** entre :

- la sortie du réseau $s_\theta(x_t, t)$ (score estimé),  
- le score « oracle » $\nabla_{x_t} \log p_t^0(x_t \mid x_0)$ calculable analytiquement puisque $p_t^0(x_t \mid x_0)$ est gaussien connu (forward SDE). 
`Intuition` : 

on se sert de  comme d’un guide qui donne la bonne direction de gradient, et le réseau apprend à l’imiter pour tous les niveaux de bruit t.
`Intuition opérationnelle` :

En pratique, pour chaque batch :

1. On échantillonne $t$, on prend des $x_0$ du dataset. 
2. On génère $x_t$ en ajoutant du bruit suivant le forward SDE (ou la formule fermée de noising). 
3. On calcule la cible $\nabla_{x_t} \log p_t^0(x_t \mid x_0)$ (souvent réécrite via le bruit $\epsilon$ ou une combinaison de $x_t$ et $x_0$).
4. On fait une MSE entre cette cible et $s_\theta(x_t, t)$, puis on moyenne.

Résultat : le réseau apprend, pour chaque $t$, à dire « dans quelle direction il faut bouger $x_t$ pour remonter la densité », ce qui sera ensuite utilisé dans le reverse SDE / probability flow ODE pour générer des échantillons. 
:::

:::
`En résumé « un modèle de diffusion =` 
- `(i)` un forward SDE qui bruit l’image, 
- `(ii)` un reverse SDE qui en dépend via le score $\nabla \log p_t$, 
- `(iii)` un réseau entraîné par score matching pour approximer ce score, permet de sampler en suivant la SDE inverse.
:::
:::
##### __`2.4. Interprétation globale`__
En résumé, à chaque petit pas $dt$ quand on remonte le temps :  
- le drift ramène $x$ vers 0 et en même temps le pousse vers des zones de forte densité de $p_t$,  
- le diffusion ajoute un peu de bruit contrôlé par $\beta(t)$
__`En pratique :`__ 
- on échantillonne $t \sim \mathcal{U}([0,T])$, $x_0$ selon la distribution de données, 
- on génère $x_t$ en ajoutant du bruit suivant le forward SDE, 
- puis on entraîne $s_\theta$ à prédire le score du bruit ajouté.
:::

:::
__`Sampling avec les modèles de diffusion (vue DDIM)`__

Dans le cas **non conditionnel**, le processus de génération par diffusion commence à partir d’un vecteur de bruit aléatoire $x_T \sim \mathcal{N}(0, I)$, puis met à jour l’état en suivant une discrétisation de l’équation de reverse SDE.

On peut aussi interpréter ce processus d’échantillonnage à la manière de **DDIM** : 
- la fonction de score joue le rôle de débruiteur et permet de prédire une version débruitée $\tilde{x}_{0|t}$ à partir d’un état quelconque $x_t$ à l’itération $t$ :
$$
\tilde{x}_{0|t}
= \frac{1}{\sqrt{\bar{\alpha}_t}}
\big(x_t + (1 - \bar{\alpha}_t)\, s_\theta(x_t, t)\big),
$$
  - où $\tilde{x}_{0|t}$ est l’estimation de $x_0$ à partir de $x_t$.

On utilise la notation standard $\alpha_t = 1 - \beta_t$ et $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ comme dans Ho et al.

Avec cette prédiction $\tilde{x}_{0|t}$ et l’état courant $x_t$, on met à jour $x_{t-1}$ via
$$
x_{t-1}
= \sqrt{\alpha_t}\,\frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\, x_t
+ \sqrt{\bar{\alpha}_{t-1}}\,\frac{\beta_t}{1 - \bar{\alpha}_t}\, \tilde{x}_{0|t}
+ \tilde{\sigma}_t z,
$$
où $z \sim \mathcal{N}(0, I)$ et $\tilde{\sigma}_t^2$ est une variance souvent fixée à $0$ (DDIM déterministe).

Le $x_{t-1}$ ainsi obtenu est réinjecté à l’itération suivante, et l’on répète cette mise à jour jusqu’à atteindre $x_0$, qui est l’image finale générée. Des détails supplémentaires sur ce schéma d’échantillonnage sont fournis dans le supplément de l’article ou dans le papier original de DDIM

:::


$dx = \left[ -\frac{\beta(t)}{2} x - \beta(t) \nabla_{x} \log p_t(x) \right] dt + \sqrt{\beta(t)} \, dw$
$\mathcal{L}(\theta) = \mathbb{E}_{x_0} \, \mathbb{E}_{x_t \mid x_0} \left\| s_\theta(x_t, t) - \nabla_{x_t} \log p_t^0(x_t \mid x_0) \right\|_2^2$
$\tilde{x}_{0|t} = \frac{1}{\sqrt{\bar{\alpha}_t}}\Big( x_t + (1 - \bar{\alpha}_t)\, s_\theta(x_t, t) \Big)$
$x_{t-1}= \sqrt{\alpha_t}\,\frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\, x_t+ \sqrt{\bar{\alpha}_{t-1}}\,\frac{\beta_t}{1 - \bar{\alpha}_t}\, \tilde{x}_{0|t}+ \tilde{\sigma}_t z$
$df_t = \left[ -\frac{\beta(t)}{2} f_t - \beta(t) \nabla_{f_t} \log p_t(f_t \mid i, v) \right] dt+ \sqrt{\beta(t)} \, dw_t$
$\nabla_{f_t} \log p_t(f_t \mid i, v)= \nabla_{f_t} \log p_t(f_t)+ \nabla_{f_t} \log p_t(i, v \mid f_t)\approx \nabla_{f_t} \log p_t(f_t)+ \nabla_{f_t} \log p_t(i, v \mid \tilde{f}_{0|t})$
$\min_{f} \, \| f - i \|_1 + \phi \, \| f - v \|_1$
$\min_{x} \, \| y - x \|_1 + \phi \, \| x \|_1$
$p(x) = \mathcal{LAP}(x; 0, \rho)= \prod_{i,j} \frac{1}{2\rho} \exp\!\left( - \frac{|x_{ij}|}{\rho} \right),p(y \mid x) = \mathcal{LAP}(y; x, \gamma)= \prod_{i,j} \frac{1}{2\gamma} \exp\!\left( - \frac{|y_{ij} - x_{ij}|}{\gamma} \right).$
$\mathcal{LAP}\!\left(\xi; \mu, \frac{p\,b}{2}\right)= \int_{0}^{\infty} \mathcal{N}(\xi; \mu, a)\,\mathcal{EXP}(a; b)\, da$
$\begin{cases}y_{ij} \mid x_{ij}, m_{ij} \sim \mathcal{N}(y_{ij}; x_{ij}, m_{ij}), \\[4pt]m_{ij} \sim \mathrm{EXP}(m_{ij}; \gamma), \\[4pt]x_{ij} \mid n_{ij} \sim \mathcal{N}(x_{ij}; 0, n_{ij}), \\[4pt]n_{ij} \sim \mathcal{EXP}(n_{ij}; \rho).\end{cases}$
$\ell(x) = \log p(x, y) - r(x)= - \sum_{i,j} \left[ \frac{(x_{ij} - y_{ij})^2}{2 m_{ij}}+ \frac{x_{ij}^2}{2 n_{ij}} \right]- \frac{\psi^2}{2} \, \|\nabla x\|_2^2$
$x^{(t+1)} = \arg\max_{x} \, Q\big(x \mid x^{(t)}\big).$

$\mathbb{E}_{m_{ij} \mid x^{(t)}_{ij}, y_{ij}}\left[ \frac{1}{m_{ij}} \right]= \sqrt{ \frac{2 (y_{ij} - x^{(t)}_{ij})^2}{\gamma} },\mathbb{E}_{n_{ij} \mid x^{(t)}_{ij}}\left[ \frac{1}{n_{ij}} \right]= \sqrt{ \frac{2 \big(x^{(t)}_{ij}\big)^2}{\rho} }.$
$\log p(\tilde{m}_{ij} \mid y_{ij}, x_{ij})= \log p(y_{ij} \mid x_{ij}, m_{ij})+ \log p(\tilde{m}_{ij})= -\frac{3}{2} \log \tilde{m}_{ij}- \tilde{m}_{ij} \frac{(y_{ij} - x_{ij})^2}{2}- \frac{1}{\gamma \tilde{m}_{ij}}+ \text{constant}.$
$p(\tilde{m}_{ij} \mid y_{ij}, x_{ij})= \mathcal{IN}\!\left(\tilde{m}_{ij};\sqrt{\frac{2 (y_{ij} - x_{ij})^2}{\gamma}},\frac{2}{\gamma}\right)$
$\log p(\tilde{n}_{ij} \mid x_{ij})= \log p(x_{ij} \mid n_{ij}) + \log p(\tilde{n}_{ij})= -\frac{3}{2} \log \tilde{n}_{ij}- \tilde{n}_{ij} \frac{x_{ij}^2}{2}- \frac{1}{\rho \tilde{n}_{ij}}+ \text{constant}.$
$p(\tilde{n}_{ij} \mid x_{ij})= \mathcal{IN}\!\left(\tilde{n}_{ij};\sqrt{\frac{2 x_{ij}^2}{\rho}},\frac{2}{\rho}\right).$
$Q = - \sum_{i,j} \left[ \bar{m}_{ij} \frac{(x_{ij} - y_{ij})^2}{2} + \bar{n}_{ij} \frac{x_{ij}^2}{2} \right] \frac{\psi^2}{2} \, \|\nabla x\|_2^2 \propto \| m^{1/2} \odot (x - y) \|_2^2 \| n^{1/2} \odot x \|_2^2 \psi \, \|\nabla x\|_2^2 $
$\min_{x, u, k} \; \| m^{1/2} \odot (x - y) \|_2^2 + \| n^{1/2} \odot x \|_2^2 + \psi \, \|u\|_2^2 \;\text{s.t.}\; u = \nabla k,\; k = x.$
$\min_{x,u,k} \; \| m^{1/2} \odot (x - y) \|_2^2 + \| n^{1/2} \odot x \|_2^2 + \psi \|u\|_2^2 + \frac{\eta}{2} \|u - \nabla k\|_2^2 + \|k - x\|_2^2$
$\min_{k} \; L_k = \|k - x\|_2^2 + \|u - \nabla k\|_2^2$

$k = \mathrm{ifft}\!\left( \dfrac{\mathrm{fft}(x) + \mathrm{fft}(\nabla) \odot \mathrm{fft}(u)}{1 + \mathrm{fft}(\nabla) \odot \mathrm{fft}(\nabla)} \right)$
$\min_{u} \; L_u = \psi \|u\|_2^2 + \frac{\eta}{2} \|u - \nabla k\|_2^2$
$u = \dfrac{\eta}{2\psi + \eta} \, \nabla k$
$\min_{x} \; L_x = \| m^{1/2} \odot (x - y) \|_2^2 + \| n^{1/2} \odot x \|_2^2 + \frac{\eta}{2} \|k - x\|_2^2$
$x = (2m^2  \odot y + ηk)  \odot (2m^2 + 2n^2 + η),$
$\hat{f} = x + v$
$\gamma = \dfrac{1}{h w} \sum_{i,j} \mathbb{E}[m_{ij}], \quad \rho = \dfrac{1}{h w} \sum_{i,j} \mathbb{E}[n_{ij}]$